# silver_transform

Lee Posts y Users de Bronze, los cura y los carga en dos tablas Delta históricas: `post_hist` y `users_hist`.
Toda la carga entra por merge, así que correrlo varias veces con los mismos datos no genera duplicados.

## Paso 1 – Librerías

In [0]:
from datetime import datetime

import pytz
from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F

## Paso 2 – Parámetros

Se pueden cambiar desde el notebook o desde el Job. Con un solo mes en `meses` se ve la diferencia entre registros nuevos y por actualizar.

In [0]:
dbutils.widgets.text("anio", "2021")
dbutils.widgets.text("meses", "1,2")
dbutils.widgets.text("storage_account", "stsbmprod001")
dbutils.widgets.text("catalogo", "silversbm")
dbutils.widgets.text("esquema", "stackoverflow")

anio = int(dbutils.widgets.get("anio"))
meses = [int(m) for m in dbutils.widgets.get("meses").split(",")]
storage_account = dbutils.widgets.get("storage_account")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")

if meses != list(range(meses[0], meses[-1] + 1)):
    raise ValueError(f"Los meses deben ser seguidos: {meses}")

bronze_base = f"abfss://bronze@{storage_account}.dfs.core.windows.net/stackoverflow"
silver_base = f"abfss://silver@{storage_account}.dfs.core.windows.net/stackoverflow"
tabla_post = f"{catalogo}.{esquema}.post_hist"
tabla_users = f"{catalogo}.{esquema}.users_hist"

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{esquema}")
print(f"Año {anio}, meses {meses}")
print(tabla_post)
print(tabla_users)

Año 2021, meses [1, 2]
silversbm.stackoverflow.post_hist
silversbm.stackoverflow.users_hist


## Paso 3 – Leer Bronze

Bronze no está en el catálogo: se lee por ruta, una carpeta por mes. Spark ignora los archivos que empiezan con `_`.

In [0]:
def leer_bronze(tabla):
    rutas = [f"{bronze_base}/{tabla}/{anio}/{mes:02d}" for mes in meses]
    return spark.read.parquet(*rutas)


leer_bronze("post").createOrReplaceTempView("bronze_post")
leer_bronze("users").createOrReplaceTempView("bronze_users")

## Paso 4 – Curar

Se quitan las columnas pesadas (`Body` y `AboutMe`) y se tipan las demás. La fuente usa 0, cadena vacía y 1970 en lugar de nulos, y aquí pasan a NULL.
Se usa `TRY_CAST` porque `ParentId` y `Reputation` vienen como texto y Serverless falla con un `CAST` de cadena vacía.

In [0]:
EPOCH = "TIMESTAMP '1970-01-01 00:00:00'"

post_curado = spark.sql(f"""
    SELECT
        Id                                       AS id,
        CAST(PostTypeId AS INT)                  AS post_type_id,
        NULLIF(AcceptedAnswerId, 0)              AS accepted_answer_id,
        NULLIF(TRY_CAST(ParentId AS BIGINT), 0)  AS parent_id,
        NULLIF(OwnerUserId, 0)                   AS owner_user_id,
        CreationDate                             AS creation_date,
        NULLIF(LastEditDate, {EPOCH})            AS last_edit_date,
        NULLIF(LastActivityDate, {EPOCH})        AS last_activity_date,
        NULLIF(ClosedDate, {EPOCH})              AS closed_date,
        CAST(Score AS INT)                       AS score,
        CAST(ViewCount AS INT)                   AS view_count,
        CAST(AnswerCount AS INT)                 AS answer_count,
        CAST(CommentCount AS INT)                AS comment_count,
        NULLIF(Title, '')                        AS title,
        NULLIF(Tags, '')                         AS tags,
        ContentLicense                           AS content_license,
        YEAR(CreationDate)                       AS post_year,
        MONTH(CreationDate)                      AS post_month
    FROM bronze_post
    WHERE Id IS NOT NULL
""")

users_curado = spark.sql(f"""
    SELECT
        Id                                       AS id,
        TRY_CAST(Reputation AS BIGINT)           AS reputation,
        CreationDate                             AS creation_date,
        NULLIF(DisplayName, '')                  AS display_name,
        NULLIF(LastAccessDate, {EPOCH})          AS last_access_date,
        CAST(Views AS INT)                       AS views,
        CAST(UpVotes AS INT)                     AS up_votes,
        CAST(DownVotes AS INT)                   AS down_votes,
        NULLIF(WebsiteUrl, '')                   AS website_url,
        NULLIF(Location, '')                     AS location,
        AccountId                                AS account_id,
        YEAR(CreationDate)                       AS creation_year,
        MONTH(CreationDate)                      AS creation_month
    FROM bronze_users
    WHERE Id IS NOT NULL
""")

## Paso 5 – Quitar duplicados

Delta falla si dos filas del origen coinciden con la misma del destino, así que por cada `id` se deja la versión más reciente.

In [0]:
def preparar(df, columna_reciente):
    ventana = Window.partitionBy("id").orderBy(F.col(columna_reciente).desc_nulls_last())
    return (
        df.withColumn("fila", F.row_number().over(ventana))
        .filter("fila = 1")
        .drop("fila")
        .withColumn("fecha_cargue", F.current_timestamp())
    )


post_origen = preparar(post_curado, "last_activity_date")
users_origen = preparar(users_curado, "last_access_date")

print(f"post: {post_curado.count():,} leídos, {post_origen.count():,} sin duplicados")
print(f"users: {users_curado.count():,} leídos, {users_origen.count():,} sin duplicados")

post: 630,020 leídos, 630,020 sin duplicados
users: 375,129 leídos, 375,129 sin duplicados


## Paso 6 – Validar y hacer merge

La tabla se crea vacía y externa la primera vez, para que toda la carga entre por merge y DuckDB la pueda leer por su ruta.
Antes del merge se cuentan los registros nuevos y por actualizar. Después se comparan con las métricas que registró Delta.

In [0]:
def cargar(df, tabla, ruta, particion):
    llaves = ["id", *particion]

    if not spark.catalog.tableExists(tabla):
        (df.limit(0).write.format("delta")
            .partitionBy(*particion)
            .option("path", ruta)
            .saveAsTable(tabla))

    filas_antes = spark.table(tabla).count()
    llaves_origen = df.select(*llaves)
    llaves_destino = spark.table(tabla).select(*llaves)
    por_actualizar = llaves_origen.join(llaves_destino, on=llaves, how="inner").count()
    nuevos = llaves_origen.join(llaves_destino, on=llaves, how="left_anti").count()
    print(f"{tabla}: {nuevos:,} nuevos, {por_actualizar:,} por actualizar")

    condicion = " AND ".join(f"tgt.{c} = src.{c}" for c in llaves)
    destino = DeltaTable.forName(spark, tabla)
    (destino.alias("tgt")
        .merge(df.alias("src"), condicion)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

    metricas = destino.history(10).filter("operation = 'MERGE'").orderBy(F.col("version").desc()).select("operationMetrics").first()[0]
    insertados = int(metricas.get("numTargetRowsInserted", 0))
    actualizados = int(metricas.get("numTargetRowsUpdated", 0))
    filas_despues = spark.table(tabla).count()
    print(f"{tabla}: merge con {insertados:,} insertados y {actualizados:,} actualizados, {filas_despues:,} filas en total")

    if insertados != nuevos or actualizados != por_actualizar:
        raise ValueError(f"{tabla}: el merge no coincide con la validación")
    if filas_despues != filas_antes + nuevos:
        raise ValueError(f"{tabla}: tenía {filas_antes} filas y quedó con {filas_despues}, con {nuevos} nuevos")

    return tabla, nuevos, por_actualizar, filas_despues


resumen = [
    cargar(post_origen, tabla_post, f"{silver_base}/post_hist", ["post_year", "post_month"]),
    cargar(users_origen, tabla_users, f"{silver_base}/users_hist", ["creation_year", "creation_month"]),
]
display(spark.createDataFrame(resumen, ["tabla", "nuevos", "por_actualizar", "filas_totales"]))

silversbm.stackoverflow.post_hist: 0 nuevos, 630,020 por actualizar
silversbm.stackoverflow.post_hist: merge con 0 insertados y 630,020 actualizados, 630,020 filas en total
silversbm.stackoverflow.users_hist: 0 nuevos, 375,129 por actualizar
silversbm.stackoverflow.users_hist: merge con 0 insertados y 375,129 actualizados, 375,129 filas en total


tabla,nuevos,por_actualizar,filas_totales
silversbm.stackoverflow.post_hist,0,630020,630020
silversbm.stackoverflow.users_hist,0,375129,375129


## Paso 7 – Revisar el resultado

Filas por mes, llaves repetidas e historial de Delta. El historial y `_delta_log` son la metadata de la tabla.

In [0]:
display(spark.sql(f"""
    SELECT post_year AS anio, post_month AS mes, COUNT(*) AS filas, COUNT(DISTINCT id) AS ids, MAX(fecha_cargue) AS fecha_cargue
    FROM {tabla_post}
    GROUP BY 1, 2
    ORDER BY 1, 2
"""))

display(spark.sql(f"""
    SELECT creation_year AS anio, creation_month AS mes, COUNT(*) AS filas, COUNT(DISTINCT id) AS ids, MAX(fecha_cargue) AS fecha_cargue
    FROM {tabla_users}
    GROUP BY 1, 2
    ORDER BY 1, 2
"""))

anio,mes,filas,ids,fecha_cargue
2021,1,325977,325977,2026-11-05T14:13:04.318Z
2021,2,304043,304043,2026-11-05T14:13:04.318Z


anio,mes,filas,ids,fecha_cargue
2021,1,194555,194555,2026-11-05T14:13:12.815Z
2021,2,180574,180574,2026-11-05T14:13:12.815Z


In [0]:
for tabla in [tabla_post, tabla_users]:
    repetidos = spark.table(tabla).groupBy("id").count().filter("count > 1").count()
    print(f"{tabla}: {repetidos} ids repetidos")
    if repetidos > 0:
        raise ValueError(f"{tabla} tiene ids repetidos")

silversbm.stackoverflow.post_hist: 0 ids repetidos
silversbm.stackoverflow.users_hist: 0 ids repetidos


In [0]:
for tabla in [tabla_post, tabla_users]:
    display(spark.sql(f"DESCRIBE HISTORY {tabla}").select("version", "timestamp", "operation", "operationMetrics"))

zona = pytz.timezone("America/Bogota")
print(f"Silver terminado: {datetime.now(zona).strftime('%Y-%m-%d %H:%M:%S')} (hora Colombia)")

version,timestamp,operation,operationMetrics
4,2026-11-05T14:13:08.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 38720499, numTargetBytesRemoved -> 38782400, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 630020, executionTimeMs -> 4234, materializeSourceTimeMs -> 1277, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1203, numTargetRowsUpdated -> 630020, numOutputRows -> 630020, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 630020, numTargetFilesRemoved -> 2, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1725)"
3,2026-11-05T14:08:37.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 18731442, numTargetBytesRemoved -> 18731442, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 304043, executionTimeMs -> 4019, materializeSourceTimeMs -> 756, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1297, numTargetRowsUpdated -> 304043, numOutputRows -> 304043, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 304043, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1922)"
2,2026-11-05T14:07:48.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 18731442, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 3381, materializeSourceTimeMs -> 904, numTargetRowsInserted -> 304043, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1136, numTargetRowsUpdated -> 0, numOutputRows -> 304043, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 304043, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1311)"
1,2026-11-05T14:06:22.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 20050958, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 6375, materializeSourceTimeMs -> 1601, numTargetRowsInserted -> 325977, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2795, numTargetRowsUpdated -> 0, numOutputRows -> 325977, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 325977, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1889)"
0,2026-11-05T14:06:12.000Z,CREATE TABLE AS SELECT,"Map(numFiles -> 0, numOutputRows -> 0, numOutputBytes -> 0)"


version,timestamp,operation,operationMetrics
4,2026-11-05T14:13:16.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 16967207, numTargetBytesRemoved -> 16950345, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 375129, executionTimeMs -> 3206, materializeSourceTimeMs -> 665, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1080, numTargetRowsUpdated -> 375129, numOutputRows -> 375129, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 375129, numTargetFilesRemoved -> 2, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1432)"
3,2026-11-05T14:08:45.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 8163308, numTargetBytesRemoved -> 8163308, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 180574, executionTimeMs -> 3099, materializeSourceTimeMs -> 509, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1144, numTargetRowsUpdated -> 180574, numOutputRows -> 180574, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 180574, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1420)"
2,2026-11-05T14:07:55.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 8163308, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2622, materializeSourceTimeMs -> 574, numTargetRowsInserted -> 180574, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 984, numTargetRowsUpdated -> 0, numOutputRows -> 180574, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 180574, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1040)"
1,2026-11-05T14:06:32.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 8787037, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2855, materializeSourceTimeMs -> 641, numTargetRowsInserted -> 194555, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 960, numTargetRowsUpdated -> 0, numOutputRows -> 194555, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 194555, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1220)"
0,2026-11-05T14:06:27.000Z,CREATE TABLE AS SELECT,"Map(numFiles -> 0, numOutputRows -> 0, numOutputBytes -> 0)"


Silver terminado: 2026-11-05 09:13:23 (hora Colombia)
